In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sqlite3 as sql
from dotenv import load_dotenv, dotenv_values
import os

load_dotenv(Path.cwd().parent / ".env")

True

In [13]:
data = pd.read_csv(f'{Path.cwd().parent / "companies_house_data.csv"}')

C:\Users\Ink\AppData\Local\Temp\ipykernel_42764\2840432469.py:1: DtypeWarning: Columns (0: RegAddress.POBox, 1: PreviousName_5.CONDATE, 2:  PreviousName_5.CompanyName, 3: PreviousName_6.CONDATE, 4:  PreviousName_6.CompanyName, 5: PreviousName_7.CONDATE, 6:  PreviousName_7.CompanyName, 7: PreviousName_8.CONDATE, 8:  PreviousName_8.CompanyName, 9: PreviousName_9.CONDATE, 10:  PreviousName_9.CompanyName, 11: PreviousName_10.CONDATE, 12:  PreviousName_10.CompanyName) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(f'{Path.cwd().parent / "companies_house_data.csv"}')


In [14]:
active_boys = data.loc[data["CompanyStatus"] == "Active"]
active_boys = active_boys.loc[active_boys["Accounts.AccountCategory"] != "DORMANT"]
active_boys = active_boys.loc[active_boys["CompanyName"].str.contains("LTD|LIMITED|LLP|LIMITED LIABILITY PARTNERSHIP", case=False, na=False)]
active_boys.rename(columns={" CompanyNumber": "CompanyNumber", "IncorporationDate": "DateOfCreation","RegAddress.PostTown":"Town", "RegAddress.County":"County", "RegAddress.PostCode": "PostCode"}, inplace=True)

In [15]:
active_boys.iloc[500]

CompanyName                                                            0 QUARTERS LTD
CompanyNumber                                                                16710077
RegAddress.CareOf                                                                 NaN
RegAddress.POBox                                                                  NaN
RegAddress.AddressLine1                                                   2 EAST ROAD
 RegAddress.AddressLine2                                                          NaN
Town                                                                           LONDON
County                                                                            NaN
RegAddress.Country                                                            ENGLAND
PostCode                                                                      E15 3QR
CompanyCategory                                               Private Limited Company
CompanyStatus                                         

In [16]:
active_boys["CompanyNumber"]

1          16092999
3          16873705
4          15073164
5          13522064
6          11006939
             ...   
5698264    14418099
5698267    14079291
5698268    11044986
5698271    09511422
5698272    11457383
Name: CompanyNumber, Length: 4390812, dtype: str

In [17]:
active_boys.iloc[100]

CompanyName                                                           $10 CHIMP LIMITED
CompanyNumber                                                                  09646807
RegAddress.CareOf                                                                   NaN
RegAddress.POBox                                                                    NaN
RegAddress.AddressLine1                                              63 GRANTHAM AVENUE
 RegAddress.AddressLine2                                                  GREAT CORNARD
Town                                                                            SUDBURY
County                                                                              NaN
RegAddress.Country                                                              ENGLAND
PostCode                                                                       CO10 0ZG
CompanyCategory                                                 Private Limited Company
CompanyStatus                   

In [18]:
active_names = active_boys[["CompanyName", "CompanyNumber","DateOfCreation","CompanyCategory", "Town","County","PostCode","SICCode.SicText_1","SICCode.SicText_2","SICCode.SicText_3","SICCode.SicText_4"]]

In [12]:
active_names

,CompanyName,CompanyNumber,DateOfCreation,CompanyCategory,Town,County,PostCode,SICCode.SicText_1,SICCode.SicText_2,SICCode.SicText_3,SICCode.SicText_4
0,! LTD,08209948,11/09/2012,Private Limited Company,HARROGATE,NaN,HG1 1ND,99999 - Dormant Company,NaN,NaN,NaN
1,!ABRIDGE TAX LTD,16092999,21/11/2024,Private Limited Company,HATFIELD,NaN,AL9 5BL,62020 - Information technology consultancy act...,NaN,NaN,NaN
2,!BIG IMPACT GRAPHICS LIMITED,11743365,28/12/2018,Private Limited Company,LONDON,NaN,EC1V 2NX,59112 - Video production activities,63120 - Web portals,74100 - specialised design activities,74201 - Portrait photographic activities
3,!FE NETWORK LIMITED,16873705,25/11/2025,Private Limited Company,"HARLOW, ESSEX",NaN,CM20 1YS,86210 - General medical practice activities,NaN,NaN,NaN
4,!NFLECTION ADVISORY LIMITED,15073164,15/08/2023,Private Limited Company,POTTERS BAR,HERTFORDSHIRE,EN6 2DA,70229 - Management consultancy activities othe...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
5698269,“NEIGH”OMI’S EQUINE TRANSPORT AND HIRE LTD,11224854,26/02/2018,Private Limited Company,ESSEX,NaN,IG1 3AD,52290 - Other transportation support activities,NaN,NaN,NaN
5698270,“Q-STAR TELENERGY HOLDINGS LIMITED,SC655320,21/02/2020,Private Limited Company,RENFREWSHIRE,NaN,PA4 8SG,61200 - Wireless telecommunications activities,61300 - Satellite telecommunications activities,61900 - Other telecommunications activities,62090 - Other information technology service a...
5698271,“SAIL IN GREECE ADVENTURES” LTD,09511422,26/03/2015,Private Limited Company,LONDON,LONDON,EC3V 3NG,79120 - Tour operator activities,NaN,NaN,NaN
5698272,“THE GREENHOUSE” COMPANY LTD,11457383,10/07/2018,Private Limited Company,SHREWSBURY,SHROPSHIRE,SY1 1XF,56101 - Licensed restaurants,NaN,NaN,NaN


In [19]:
active_names["CompanyName"] = active_names["CompanyName"].str.lstrip("!") 

In [20]:
active_names

,CompanyName,CompanyNumber,DateOfCreation,CompanyCategory,Town,County,PostCode,SICCode.SicText_1,SICCode.SicText_2,SICCode.SicText_3,SICCode.SicText_4
1,ABRIDGE TAX LTD,16092999,21/11/2024,Private Limited Company,HATFIELD,NaN,AL9 5BL,62020 - Information technology consultancy act...,NaN,NaN,NaN
3,FE NETWORK LIMITED,16873705,25/11/2025,Private Limited Company,"HARLOW, ESSEX",NaN,CM20 1YS,86210 - General medical practice activities,NaN,NaN,NaN
4,NFLECTION ADVISORY LIMITED,15073164,15/08/2023,Private Limited Company,POTTERS BAR,HERTFORDSHIRE,EN6 2DA,70229 - Management consultancy activities othe...,NaN,NaN,NaN
5,NFOGENIE LTD,13522064,21/07/2021,Private Limited Company,LONDON,GREATER LONDON,WC2H 9JQ,58290 - Other software publishing,NaN,NaN,NaN
6,NNOV8 LIMITED,11006939,11/10/2017,Private Limited Company,EDENBRIDGE,NaN,TN8 5NF,62090 - Other information technology service a...,70229 - Management consultancy activities othe...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
5698264,‘HEOL CHWARAE RÔL LTD,14418099,13/10/2022,"PRI/LTD BY GUAR/NSC (Private, limited by guara...",PONTYPRIDD,NaN,CF38 1TQ,93290 - Other amusement and recreation activit...,NaN,NaN,NaN
5698267,"“34 HALLAM ROAD FLAT MANAGEMENT COMPANY"" LIMITED",14079291,29/04/2022,Private Limited Company,CLEVEDON,NaN,BS21 7NE,99999 - Dormant Company,NaN,NaN,NaN
5698268,“ARTIS UK” LIMITED,11044986,02/11/2017,Private Limited Company,SURBITON,NaN,KT6 5LX,41202 - Construction of domestic buildings,NaN,NaN,NaN
5698271,“SAIL IN GREECE ADVENTURES” LTD,09511422,26/03/2015,Private Limited Company,LONDON,LONDON,EC3V 3NG,79120 - Tour operator activities,NaN,NaN,NaN


In [21]:
def add_company_names_to_db(df):
    con = sql.connect(Path.cwd().parent / os.getenv("DB_URL"))
    cur = con.cursor()

    for index, row in df.iterrows():
        cur.execute("INSERT INTO companies (name, company_id, date_of_creation, company_type, town, county, post_code) VALUES (?, ?, ?, ?, ?, ?, ?)", (row["CompanyName"], row["CompanyNumber"], row["DateOfCreation"], row["CompanyCategory"], row["Town"], row["County"], row["PostCode"]))
    con.commit()
    con.close()
    
def add_sic_codes_to_db(df):
    con = sql.connect(Path.cwd().parent / os.getenv("DB_URL"))
    cur = con.cursor()

    for index, row in df.iterrows():
        sic_codes = row[["SICCode.SicText_1", "SICCode.SicText_2", "SICCode.SicText_3", "SICCode.SicText_4"]].dropna().tolist()
        if 'None Supplied' in sic_codes:
            sic_codes.remove('None Supplied')
        for i in sic_codes:
            code, text = i.split(" - ")
            try:
                cur.execute("INSERT INTO sic_codes (sic_id, sector_name) VALUES (?, ?)", (code.strip(), text.strip()))
            except sql.IntegrityError:
                continue
    
    con.commit()
    con.close()        
    
def add_company_sic_codes_to_db(df):
    con = sql.connect(Path.cwd().parent / os.getenv("DB_URL"))
    cur = con.cursor()

    for index, row in df.iterrows():
        company_id = row["CompanyNumber"]
        sic_codes = row[["SICCode.SicText_1", "SICCode.SicText_2", "SICCode.SicText_3", "SICCode.SicText_4"]].dropna().tolist()
        if 'None Supplied' in sic_codes:
            sic_codes.remove('None Supplied')
        for i in sic_codes:
            code, text = i.split(" - ")
            try:
                cur.execute("INSERT INTO companies_sic_codes (company_id, sic_code) VALUES (?, ?)", (company_id, code.strip()))

            except sql.IntegrityError:
                continue

    con.commit()
    con.close()

In [22]:
add_company_names_to_db(active_names)

In [52]:
add_sic_codes_to_db(active_names)

In [23]:
add_company_sic_codes_to_db(active_names)

In [22]:
sic_codes = data.iloc[-1][["SICCode.SicText_1", "SICCode.SicText_2", "SICCode.SicText_3", "SICCode.SicText_4"]].dropna().tolist()

In [24]:
sic_codes.remove('None Supplied')

In [62]:
raw_val = pd.read_csv(f'{Path.cwd() / "com_names_sme 1.csv"}', dtype={"com_num":str})

In [63]:
print(raw_val["com_num"].to_list())

['15073164', '13522064', '11006939', 'SC606050', '11303802', '05914136', '04494986', '02537158', 'SC313991', '02871100', '01382560', '04427981', '14502415', '02591631', '14428804', '15081740', '05210908', '09102217', '00829173', '01941022', '01952156', '00851579', '08283517', '10339143', '03606467', '02318517', '01852173', '13138542', '00805326', '13802242', '02184955', '09536715', 'SC412218', '02407980', '10480357', '01030234', '04635689', '03093541', '07909447', '09712492', '00263728', '06991687', '03242800', '03079224', '06383456', '07051447', '01816008', '05179519', '06122882', '00899832', '02396957', '10561154', '01636551', '00641164', '02408030', '13784047', '05180993', '00784285', '02306031', '02302474', '04372047', 'SC287325', '04656696', '07390480', '07390512', '15092367', '09646807', '15884314', '13984698', '13925097', '14624244', '15762125', '13315396', '15295014', '10715295', '14652281', '15103868', '08684770', '13014456', '13261308', '11126054', '13999081', '08697873', '14

In [64]:
result = []
for i in raw_val["com_num"]:
    conn = sql.connect(Path.cwd().parent / os.getenv("DB_URL"))
    cursor = conn.cursor()

    cursor.execute("SELECT c.company_type, c.town, c.county, c.post_code FROM companies c WHERE c.company_id = ?", (i,))
    out = cursor.fetchall()

    com_type, town, county, post_code = out[0] if out else (None, None, None, None)
    result.append((com_type, town, county, post_code))

result

[('Private Limited Company', 'POTTERS BAR', 'HERTFORDSHIRE', 'EN6 2DA'),
 ('Private Limited Company', 'LONDON', 'GREATER LONDON', 'WC2H 9JQ'),
 ('Private Limited Company', 'EDENBRIDGE', None, 'TN8 5NF'),
 ('Private Limited Company', 'ABERDEEN', None, 'AB11 7SY'),
 ('Private Limited Company', 'GREAT WAKERING', 'ESSEX', 'SS3 0GW'),
 ('PRI/LTD BY GUAR/NSC (Private, limited by guarantee, no share capital)',
  'CROYDON',
  'SURREY',
  'CR0 2RF'),
 ('Private Limited Company', 'BOLTON', None, 'BL2 1HQ'),
 ('Private Limited Company', 'LONDON', None, 'E1 7AA'),
 ('Private Limited Company', 'GLENROTHES', None, 'KY6 2RU'),
 ('Private Limited Company', None, None, 'W12 8LA'),
 ('Private Limited Company', 'NORWICH', None, 'NR2 2AL'),
 ('Private Limited Company', 'WISBECH', 'CAMBRIDGESHIRE', 'PE13 1EH'),
 ('Private Limited Company', 'CORBY', None, 'NN17 2JY'),
 ('PRI/LTD BY GUAR/NSC (Private, limited by guarantee, no share capital)',
  'ST ASAPH',
  'DENBIGHSHIRE',
  'LL17 0RE'),
 ('Private Limited 

In [65]:
add_on = pd.DataFrame(result, columns=["company_type", "town", "county", "post_code"])
add_on

,company_type,town,county,post_code
0,Private Limited Company,POTTERS BAR,HERTFORDSHIRE,EN6 2DA
1,Private Limited Company,LONDON,GREATER LONDON,WC2H 9JQ
2,Private Limited Company,EDENBRIDGE,NaN,TN8 5NF
3,Private Limited Company,ABERDEEN,NaN,AB11 7SY
4,Private Limited Company,GREAT WAKERING,ESSEX,SS3 0GW
...,...,...,...,...
13539,"PRI/LTD BY GUAR/NSC (Private, limited by guara...",BERKHAMSTED,NaN,HP4 2EG
13540,Private Limited Company,LONDON,NaN,E6 3RX
13541,Private Limited Company,LEICESTER,NaN,LE7 7NF
13542,Private Limited Company,LIVERPOOL,NaN,L23 5RE


In [67]:
pd.concat([raw_val, add_on], axis=1).to_csv(f'{Path.cwd() / "6_7_2026_out.csv"}', index=False)

In [68]:
results

[('Private Limited Company', 'CRANLEIGH', 'SURREY', 'GU6 7FL')]

In [69]:
data = pd.DataFrame(results, columns=["com_num","name", "sic_code"])

ValueError: 3 columns passed, passed data had 4 columns

In [70]:
final = data.iloc[30000:40000]
index = "[30000-40000]"

In [18]:
final

,com_num,name,sic_code
30000,11267284,20 NS MANAGEMENT LIMITED,68209
30001,05686727,20 OAKSFORD AVENUE ENFRANCHISEMENT COMPANY LIM...,55900
30002,07217524,20 OBAN ROAD LIMITED,68100
30003,SC846178,20 OF LTD,73110
30004,SC846178,20 OF LTD,82990
...,...,...,...
39995,16352322,2717 PRESERVATION GROUP LIMITED,33170
39996,16352322,2717 PRESERVATION GROUP LIMITED,49390
39997,16352322,2717 PRESERVATION GROUP LIMITED,91020
39998,16352322,2717 PRESERVATION GROUP LIMITED,94990


In [19]:
final.to_csv(Path.cwd().parent / f"com_names_n_codes{index}.csv", index=False)

In [49]:
data[data["CompanyName"] == '"243 RUGBY ROAD MANAGEMENT COMPANY LIMITED"']

,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_7.CONDATE,PreviousName_7.CompanyName,PreviousName_8.CONDATE,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate
13,"""243 RUGBY ROAD MANAGEMENT COMPANY LIMITED""",05914136,NaN,NaN,STONEMEAD HOUSE,95 LONDON ROAD,CROYDON,SURREY,ENGLAND,CR0 2RF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,06/09/2026,23/08/2025
